# 04 — Training & Evaluation

| Script | Contents |
|---|---|
| `train.py` | Full training loop (`train`, `train_all_intents`) |
| `main.py` | Demo, compare, and analyze modes |

This notebook trains a defender from scratch, then runs the three evaluation
modes: **demo**, **compare across intents**, and **analyze transition matrices**.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid')

---
## 1. Training (`train.py`)

The `train()` function runs a full DQN training loop:
1. Optionally fits the RandomForest attack classifier
2. Runs N episodes with epsilon-greedy exploration
3. Evaluates periodically (greedy policy) and saves checkpoints

> **Tip**: set `n_episodes=50` for a quick smoke-test; use 300+ for real convergence.

In [ ]:
from train import train
from attacker.attack_types import AttackerIntent

metrics = train(
    n_episodes       = 100,          # increase for better convergence
    max_steps        = 300,
    intent           = AttackerIntent.OPPORTUNISTIC,
    save_dir         = '../models/',
    log_dir          = '../logs/train_nb/',
    seed             = 42,
    eval_interval    = 25,
    save_interval    = 50,
    train_classifier = True,
    n_clf_samples    = 400,
    verbose          = True,
)

### 1.1 Training curves

In [ ]:
ep_df = pd.DataFrame([
    {
        'episode':             e.episode,
        'total_reward':        e.total_reward,
        'detection_rate':      e.detection_rate,
        'false_positive_rate': e.false_positive_rate,
        'avg_threat_level':    e.avg_threat_level,
        'avg_loss':            e.avg_loss,
    }
    for e in metrics.episodes
])

def rolling(series, w=10):
    return series.rolling(w, min_periods=1).mean()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Training Curves — OPPORTUNISTIC attacker', fontsize=13, fontweight='bold')

def plot_curve(ax, x, y, label, color, ylim=None):
    ax.plot(x, y, alpha=0.35, color=color)
    ax.plot(x, rolling(pd.Series(y)), color=color, linewidth=2, label='Rolling(10)')
    ax.set_title(label)
    ax.set_xlabel('Episode')
    ax.grid(True, alpha=0.3)
    if ylim:
        ax.set_ylim(*ylim)
    ax.legend(fontsize=8)

plot_curve(axes[0,0], ep_df.episode, ep_df.total_reward,        'Total Reward',        'steelblue')
plot_curve(axes[0,1], ep_df.episode, ep_df.detection_rate,      'Detection Rate',      'green',  (0,1))
plot_curve(axes[0,2], ep_df.episode, ep_df.false_positive_rate, 'False Positive Rate', 'red',    (0,1))
plot_curve(axes[1,0], ep_df.episode, ep_df.avg_threat_level,    'Avg Threat Level',    'orange', (0,1))
plot_curve(axes[1,1], ep_df.episode, ep_df.avg_loss,            'Avg DQN Loss',        'purple')

# Epsilon schedule approximation
eps_vals = [max(0.05, 1.0 * (0.997 ** (i * 300))) for i in range(len(ep_df))]
axes[1,2].plot(ep_df.episode, eps_vals, color='teal')
axes[1,2].set_title('Epsilon (approx.)')
axes[1,2].set_xlabel('Episode')
axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. Demo Mode (`main.py — run_demo`)

Loads the trained model and runs one full episode, printing step-by-step output
and saving an attack progression plot.

In [ ]:
from main import run_demo

run_demo(
    model_dir   = '../models/',
    intent_name = 'OPPORTUNISTIC',
    n_steps     = 100,
    render      = True,
    save_plot   = True,
    log_dir     = '../logs/demo_nb/',
)

In [ ]:
# Display the saved progression plot inline
from IPython.display import Image
plot_path = '../logs/demo_nb/demo_progression.png'
if os.path.exists(plot_path):
    Image(plot_path)
else:
    print('Plot not found — run the cell above first.')

---
## 3. Compare Mode (`main.py — run_compare`)

Evaluates the trained policy against **all 4 attacker intents** and prints a
summary table of mean reward, detection rate, and false positive rate.

In [ ]:
from main import run_compare

run_compare(
    model_dir  = '../models/',
    n_episodes = 3,
    n_steps    = 200,
    log_dir    = '../logs/compare_nb/',
)

---
## 4. Analyze Mode (`main.py — analyze`)

Visualizes all intent-specific transition matrices and feature distributions
without requiring a trained model.

In [ ]:
from main import analyze

analyze(log_dir='../logs/analyze_nb/')

In [ ]:
from IPython.display import Image

for fname in ['transition_matrices.png', 'feature_distributions.png']:
    path = f'../logs/analyze_nb/{fname}'
    if os.path.exists(path):
        display(Image(path))

---
## 5. Multi-intent training (`train.py — train_all_intents`)

Train a separate defender for each intent and print a comparison table.  
**Warning**: this runs 4 × `n_episodes` training loops — reduce `n_episodes` for quick tests.

In [ ]:
# Uncomment to run multi-intent training (slow)
# from train import train_all_intents
#
# all_results = train_all_intents(
#     n_episodes    = 50,
#     max_steps     = 200,
#     seed          = 42,
#     base_log_dir  = '../logs/multi_intent/',
#     base_save_dir = '../models/multi_intent/',
# )